In [2]:
import pandas as pd

df = pd.read_csv(
    "apartments_for_rent_classified_100K.csv",
    sep=";",
    encoding="cp1252"
)

print(df.shape)
print(df.columns.tolist())
df.head(2)

(99492, 22)
['id', 'category', 'title', 'body', 'amenities', 'bathrooms', 'bedrooms', 'currency', 'fee', 'has_photo', 'pets_allowed', 'price', 'price_display', 'price_type', 'square_feet', 'address', 'cityname', 'state', 'latitude', 'longitude', 'source', 'time']


C:\Users\asus\AppData\Local\Temp\ipykernel_23708\1758057507.py:3: DtypeWarning: Columns (0: address) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,id,category,title,body,amenities,bathrooms,bedrooms,currency,fee,has_photo,...,price_display,price_type,square_feet,address,cityname,state,latitude,longitude,source,time
0,5668640009,housing/rent/apartment,One BR 507 & 509 Esplanade,"This unit is located at 507 & 509 Esplanade, R...",NaN,1.0,1.0,USD,No,Thumbnail,...,"$2,195",Monthly,542,507 509 Esplanade,Redondo Beach,CA,33.8520,-118.3759,RentLingo,1577360355
1,5668639818,housing/rent/apartment,Three BR 146 Lochview Drive,"This unit is located at 146 Lochview Drive, Ne...",NaN,1.5,3.0,USD,No,Thumbnail,...,"$1,250",Monthly,1500,146 Lochview Dr,Newport News,VA,37.0867,-76.4941,RentLingo,1577360340


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df2 = df.copy()

# drop missing target/text
df2 = df2.dropna(subset=["bedrooms", "title", "body"]).copy()

# bedrooms -> int, filter kelas ekstrem
df2["bedrooms"] = pd.to_numeric(df2["bedrooms"], errors="coerce")
df2 = df2.dropna(subset=["bedrooms"])
df2["bedrooms"] = df2["bedrooms"].astype(int)
df2 = df2[(df2["bedrooms"] >= 0) & (df2["bedrooms"] <= 6)].copy()

# encode target -> 0..K-1
y_le = LabelEncoder()
y = y_le.fit_transform(df2["bedrooms"].astype(str))
n_classes = len(np.unique(y))
print("Classes:", list(y_le.classes_), "n_classes:", n_classes)

train_idx, test_idx = train_test_split(
    np.arange(len(df2)),
    test_size=0.2,
    random_state=42,
    stratify=y
)

df_train = df2.iloc[train_idx].reset_index(drop=True)
df_test  = df2.iloc[test_idx].reset_index(drop=True)
y_train = y[train_idx]
y_test  = y[test_idx]

Classes: ['0', '1', '2', '3', '4', '5', '6'] n_classes: 7


Neural Network: Text CNN (title + body)

In [5]:
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score

# ===== 1) Build text tensors (FIX: no dtype=object) =====
text_train = (df_train["title"].fillna("") + " " + df_train["body"].fillna("")).astype(str).str.lower().tolist()
text_test  = (df_test["title"].fillna("")  + " " + df_test["body"].fillna("")).astype(str).str.lower().tolist()

text_train_tf = tf.constant(text_train, dtype=tf.string)
text_test_tf  = tf.constant(text_test, dtype=tf.string)

# ===== 2) Vectorizer =====
max_tokens = 40000
seq_len = 200

vectorizer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=seq_len
)
vectorizer.adapt(text_train_tf)

# ===== 3) Text CNN Model =====
embed_dim = 128

inputs = keras.Input(shape=(1,), dtype=tf.string, name="text")
x = vectorizer(inputs)
x = layers.Embedding(input_dim=max_tokens, output_dim=embed_dim, mask_zero=True)(x)

x = layers.Conv1D(128, kernel_size=5, activation="relu", padding="same")(x)
x = layers.MaxPooling1D(pool_size=2)(x)
x = layers.Conv1D(128, kernel_size=5, activation="relu", padding="same")(x)
x = layers.GlobalMaxPooling1D()(x)

x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(n_classes, activation="softmax")(x)

text_cnn = keras.Model(inputs, outputs, name="text_cnn")

text_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]

text_cnn.summary()

# ===== 4) Train + runtime =====
start = time.time()
text_cnn.fit(
    text_train_tf, y_train,
    validation_split=0.1,
    epochs=15,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)
runtime_train_text = time.time() - start

# ===== 5) Predict + runtime =====
start = time.time()
pred_prob = text_cnn.predict(text_test_tf, verbose=0)
pred_text = pred_prob.argmax(axis=1)
runtime_pred_text = time.time() - start

acc_text = accuracy_score(y_test, pred_text)

print(f"\nText CNN Accuracy: {acc_text:.4f}")
print(f"Text CNN Runtime (train): {runtime_train_text:.2f}s")
print(f"Text CNN Runtime (predict): {runtime_pred_text:.2f}s")

d:\Semester 6\SC\tugas 2\.venv\Lib\site-packages\keras\src\layers\layer.py:982: UserWarning: Layer 'conv1d_2' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "text_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text (InputLayer)               │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 200)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 200, 128)       │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 200, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 100, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,301,511 (20.22 MB)

 Trainable params: 5,301,511 (20.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
280/280 ━━━━━━━━━━━━━━━━━━━━ 69s 232ms/step - accuracy: 0.7443 - loss: 0.5866 - val_accuracy: 0.8388 - val_loss: 0.3640 - learning_rate: 0.0010
Epoch 2/15
280/280 ━━━━━━━━━━━━━━━━━━━━ 63s 226ms/step - accuracy: 0.8627 - loss: 0.3218 - val_accuracy: 0.8507 - val_loss: 0.3395 - learning_rate: 0.0010
Epoch 3/15
280/280 ━━━━━━━━━━━━━━━━━━━━ 62s 220ms/step - accuracy: 0.8931 - loss: 0.2554 - val_accuracy: 0.8472 - val_loss: 0.3437 - learning_rate: 0.0010
Epoch 4/15
280/280 ━━━━━━━━━━━━━━━━━━━━ 62s 220ms/step - accuracy: 0.9192 - loss: 0.1991 - val_accuracy: 0.8499 - val_loss: 0.3800 - learning_rate: 0.0010
Epoch 5/15
280/280 ━━━━━━━━━━━━━━━━━━━━ 59s 209ms/step - accuracy: 0.9504 - loss: 0.1287 - val_accuracy: 0.8451 - val_loss: 0.4576 - learning_rate: 5.0000e-04

Text CNN Accuracy: 0.8512
Text CNN Runtime (train): 315.57s
Text CNN Runtime (predict): 11.11s
